# VisiPruner trên SmolVLM · Phần 1/2

Bật GPU và Internet. Phần này chạy cấu hình **01–12** trên 4.204 câu test. Log và kết quả lưu theo từng bộ ảnh, nên có thể chạy lại cell của cấu hình đang dở.

In [ ]:
%%writefile /kaggle/working/visipruner-ablation.env
REPOSITORY=https://github.com/huyvanzzz/Log_Metrics.git
SOURCE_BRANCH=smolvlm-attention-nguyet
SOURCE_DIR=/kaggle/working/log-metrics-visipruner
DATA_DIR=/kaggle/working/drivelm-nuscenes-v1-0
export HF_TOKEN_SECRET=''
OUTPUT_DIR=/kaggle/working/visipruner_ablation_outputs
BATCH_SIZE=8
MAX_NEW_TOKENS=128


## Cài đặt

Cập nhật mã nguồn và cài thư viện chạy mô hình.

In [ ]:
%%bash
set -euo pipefail
source /kaggle/working/visipruner-ablation.env
if [ -d "$SOURCE_DIR/.git" ]; then
  test "$(git -C "$SOURCE_DIR" branch --show-current)" = "$SOURCE_BRANCH"
  test -z "$(git -C "$SOURCE_DIR" status --porcelain --untracked-files=no)"
  git -C "$SOURCE_DIR" pull --ff-only origin "$SOURCE_BRANCH"
else
  git clone --depth 1 --single-branch --branch "$SOURCE_BRANCH" "$REPOSITORY" "$SOURCE_DIR"
fi
python -m pip install -q -r "$SOURCE_DIR/smolvlm_attention/requirements.txt"


## Chuẩn bị dữ liệu

Tải split test đã ghim, lấy mẫu phân tầng và ghi các vị trí merge vào manifest.

In [ ]:
%%bash
set -euo pipefail
source /kaggle/working/visipruner-ablation.env
cd "$SOURCE_DIR"
python -u -m smolvlm_attention.mvpruner_ablation fetch-hf --output "$DATA_DIR"
python -u -m smolvlm_attention.visipruner_ablation prepare --test "$DATA_DIR/multi_frame/multi_frame_test.json" --image-root "$DATA_DIR" --output "$OUTPUT_DIR"


## 01/23 · Baseline · SmolVLM với đủ token ảnh

Sinh đáp án baseline và lưu thứ hạng token ảnh ở các layer dùng để mask.

In [ ]:
%%bash
set -euo pipefail
source /kaggle/working/visipruner-ablation.env
cd "$SOURCE_DIR"
python -u -m smolvlm_attention.visipruner_ablation run --variant baseline --output "$OUTPUT_DIR" --image-root "$DATA_DIR" --batch-size "$BATCH_SIZE" --max-new-tokens "$MAX_NEW_TOKENS"


## 02/23 · Layer 1–2 · Mask top 10%

Can thiệp sau softmax trong prefill. Lưu tổng lượng attention bị lấy đi.

In [ ]:
%%bash
set -euo pipefail
source /kaggle/working/visipruner-ablation.env
cd "$SOURCE_DIR"
python -u -m smolvlm_attention.visipruner_ablation run --variant shallow-top10 --output "$OUTPUT_DIR" --image-root "$DATA_DIR" --batch-size "$BATCH_SIZE" --max-new-tokens "$MAX_NEW_TOKENS"


## 03/23 · Layer 1–2 · Chỉ giữ top 10%

Can thiệp sau softmax trong prefill. Lưu tổng lượng attention bị lấy đi.

In [ ]:
%%bash
set -euo pipefail
source /kaggle/working/visipruner-ablation.env
cd "$SOURCE_DIR"
python -u -m smolvlm_attention.visipruner_ablation run --variant shallow-keep10 --output "$OUTPUT_DIR" --image-root "$DATA_DIR" --batch-size "$BATCH_SIZE" --max-new-tokens "$MAX_NEW_TOKENS"


## 04/23 · Layer 1 · Mask toàn bộ attention lên ảnh

Can thiệp sau softmax trong prefill. Lưu tổng lượng attention bị lấy đi.

In [ ]:
%%bash
set -euo pipefail
source /kaggle/working/visipruner-ablation.env
cd "$SOURCE_DIR"
python -u -m smolvlm_attention.visipruner_ablation run --variant shallow-none --output "$OUTPUT_DIR" --image-root "$DATA_DIR" --batch-size "$BATCH_SIZE" --max-new-tokens "$MAX_NEW_TOKENS"


## 05/23 · Layer 1 · Merge attention với seed 42

Dồn visual attention vào patch đã chọn theo camera và vùng ảnh. Giữ nguyên tổng trọng số cho mỗi query và head.

In [ ]:
%%bash
set -euo pipefail
source /kaggle/working/visipruner-ablation.env
cd "$SOURCE_DIR"
python -u -m smolvlm_attention.visipruner_ablation run --variant shallow-merge-42 --output "$OUTPUT_DIR" --image-root "$DATA_DIR" --batch-size "$BATCH_SIZE" --max-new-tokens "$MAX_NEW_TOKENS"


## 06/23 · Layer 1 · Merge attention với seed 43

Dồn visual attention vào patch đã chọn theo camera và vùng ảnh. Giữ nguyên tổng trọng số cho mỗi query và head.

In [ ]:
%%bash
set -euo pipefail
source /kaggle/working/visipruner-ablation.env
cd "$SOURCE_DIR"
python -u -m smolvlm_attention.visipruner_ablation run --variant shallow-merge-43 --output "$OUTPUT_DIR" --image-root "$DATA_DIR" --batch-size "$BATCH_SIZE" --max-new-tokens "$MAX_NEW_TOKENS"


## 07/23 · Layer 1 · Merge attention với seed 44

Dồn visual attention vào patch đã chọn theo camera và vùng ảnh. Giữ nguyên tổng trọng số cho mỗi query và head.

In [ ]:
%%bash
set -euo pipefail
source /kaggle/working/visipruner-ablation.env
cd "$SOURCE_DIR"
python -u -m smolvlm_attention.visipruner_ablation run --variant shallow-merge-44 --output "$OUTPUT_DIR" --image-root "$DATA_DIR" --batch-size "$BATCH_SIZE" --max-new-tokens "$MAX_NEW_TOKENS"


## 08/23 · Layer 9–15 · Mask top 10%

Can thiệp sau softmax trong prefill. Lưu tổng lượng attention bị lấy đi.

In [ ]:
%%bash
set -euo pipefail
source /kaggle/working/visipruner-ablation.env
cd "$SOURCE_DIR"
python -u -m smolvlm_attention.visipruner_ablation run --variant middle-9-15-top10 --output "$OUTPUT_DIR" --image-root "$DATA_DIR" --batch-size "$BATCH_SIZE" --max-new-tokens "$MAX_NEW_TOKENS"


## 09/23 · Layer 9–15 · Mask bottom 10%

Can thiệp sau softmax trong prefill. Lưu tổng lượng attention bị lấy đi.

In [ ]:
%%bash
set -euo pipefail
source /kaggle/working/visipruner-ablation.env
cd "$SOURCE_DIR"
python -u -m smolvlm_attention.visipruner_ablation run --variant middle-9-15-bottom10 --output "$OUTPUT_DIR" --image-root "$DATA_DIR" --batch-size "$BATCH_SIZE" --max-new-tokens "$MAX_NEW_TOKENS"


## 10/23 · Layer 9–15 · Chỉ giữ top 5%

Can thiệp sau softmax trong prefill. Lưu tổng lượng attention bị lấy đi.

In [ ]:
%%bash
set -euo pipefail
source /kaggle/working/visipruner-ablation.env
cd "$SOURCE_DIR"
python -u -m smolvlm_attention.visipruner_ablation run --variant middle-9-15-keep5 --output "$OUTPUT_DIR" --image-root "$DATA_DIR" --batch-size "$BATCH_SIZE" --max-new-tokens "$MAX_NEW_TOKENS"


## 11/23 · Layer 9–15 · Mask top 10% và bảo toàn visual mass

Mask token đã chọn, sau đó phân bổ lại trọng số lên các token ảnh còn lại. Giữ nguyên trọng số lên text.

In [ ]:
%%bash
set -euo pipefail
source /kaggle/working/visipruner-ablation.env
cd "$SOURCE_DIR"
python -u -m smolvlm_attention.visipruner_ablation run --variant middle-9-15-top10-mass --output "$OUTPUT_DIR" --image-root "$DATA_DIR" --batch-size "$BATCH_SIZE" --max-new-tokens "$MAX_NEW_TOKENS"


## 12/23 · Layer 9–15 · Mask bottom 10% và bảo toàn visual mass

Mask token đã chọn, sau đó phân bổ lại trọng số lên các token ảnh còn lại. Giữ nguyên trọng số lên text.

In [ ]:
%%bash
set -euo pipefail
source /kaggle/working/visipruner-ablation.env
cd "$SOURCE_DIR"
python -u -m smolvlm_attention.visipruner_ablation run --variant middle-9-15-bottom10-mass --output "$OUTPUT_DIR" --image-root "$DATA_DIR" --batch-size "$BATCH_SIZE" --max-new-tokens "$MAX_NEW_TOKENS"


## Xác minh đầu ra để chuyển sang phần 2

Sau khi xác minh thành công, lưu output của notebook. Trong notebook phần 2, gắn output đã lưu làm Kaggle Input. Thư mục cần chuyển là `visipruner_ablation_outputs/` trong `/kaggle/working`.

In [ ]:
%%bash
set -euo pipefail
source /kaggle/working/visipruner-ablation.env
cd "$SOURCE_DIR"
python -m smolvlm_attention.visipruner_ablation handoff --output "$OUTPUT_DIR" --source "$OUTPUT_DIR"
